# Comparing Adjacency Matrices 

---

Proteomics Data Analysis Package FRAPPE: Framework for Rhythmic Analysis of Preliminary Proteomics Experiments Tools to analyze the circadian rhytmicity of proteins throughout two experimental conditions. 


* Dependencies/Imports

In [1]:
import numpy as np
import pandas as pd

* Helper Functions - Creating Correlation and Adjacency Matrices

In [ ]:
def corr_input (file): 
    """
    Finds the correlation of the protein against itself 
    in a bi-condition comparison from TLCC output. 
    -------------
    Parameters: 
    file: csv of correlation between same 
    protein of two groups 
    -------------
    Returns: corr (int): the correlation values
    """
    
    corr = pd.read_csv(file)
    corr = corr[['Corr']]

    return corr 

In [ ]:
def create_corr_matrix (data_frame, file):
    """
    Requires correlations form TLCC csv output. Creates a correlation 
    matrix for a dataframe for when the correlation is the protein against
    itself and the correlation from the file inserts the diagonal. 
    -------------
    Parameters: 
    dataframe: array of integers representing correlation
    file: csv file including correlation for protein against itself  
    -------------
    Return: corr_matrix (array): an array where protein correlation
    values are compared to itself and other protein correlation
    """
    
    corr = corr_input(file)
    corr_matrix = data_frame.T.corr(method = 'pearson')
    np.fill_diagonal(corr_matrix.values, corr)

    return corr_matrix

In [ ]:
def corr_to_adj (corr_matrix, threshold):
  """
  Creates a correlation matrix based on the given dataframe
  using Pearson Correlation coeffecient. 
  -------------
  Parameters: 
  corr_matrix: (array) array of integers representing correlation
  threshold: (int) threshold to pass/not pass rhymicity in both conditions
  ------------
  Return: adj_matrix (array): array of 1's where corr value is > then the
  threshold, and 0's where corr value is < then the threshold 
  """

  adj_matrix = corr_matrix.map(lambda x: 1 if abs(x) > threshold else 0)
  return adj_matrix


In [ ]:
def flatten_matrix (matrix): 
    """
    Flattens a matrix into a numpy list
    -------------
    Parameters: 
    matrix1: (array) adjacency matrix for one group condition
    -------------
    Returns: (array) a numpy 1D array of the upper triangle elements  
    """

    return matrix[np.triu_indices_from(matrix, k=1)]

* Helper Functions - Comparing matrices and extracting results

In [ ]:
def sub_matrices(adj_matrix1, adj_matrix2): 
    """
    Subtracts an adjacency matrix from another.
    -------------
    Parameters: 
    adj_matrix1: (array) adjacency matrix for one group condition
    adj_matrix2: (array) adjacency matrix for one group condition
    ------------
    Returns: array: the resulting adjacency matrix after
    subtracting one from another  
    """

    return (adj_matrix1 - adj_matrix2).to_numpy()

In [ ]:
def add_matrices(adj_matrix1, adj_matrix2): 
    """
    Adds an adjacency matrix to another.
    -------------
    Parameters: 
    adj_matrix1: (array) adjacency matrix for one group condition
    adj_matrix2: (array) adjacency matrix for one group condition
    -------------
    Returns: array: the resulting adjacency matrix after
    adding one to another 
    """
    return (adj_matrix1 + adj_matrix2).to_numpy()

In [ ]:
def retrieve_occurences (flat_matrix): 
    """
    Makes a dictionary of the occurences of values within a list.
    -------------
    Parameters: 
    flat_matrix: (array) adjacency matrix for one group condition
    ------------
    Returns: dictionary: key = int value, value = count of how often key 
    appears in list  
    """

    unique, counts = np.unique(flat_matrix, return_counts=True)
    return dict(zip(unique, counts))

In [ ]:
def combine_dict (dict1, dict2): 
    """
    Combines two dictionaries. 
    ------------
    Parameters: 
    dict1 :(dictionary) 1st dictionary 
    dict2 :(dictionary) 2nd dictionary  
    ------------
    Returns: dictionary: a combined dictionary of dict1 and dict2 
    """
    return dict1 | dict2

In [ ]:
def freq_dict (freq_dict, num1, num2): 
    """
    Makes a dictionary of occurences for two specific  
    value of the integers given. Num1 and Num2 should not 
    be the same integer.
    ------------
    Parameters: 
    freq_dict :(dictionary) adjacency matrix for one group condition
    num1 :(int) occurence of number given 
    num2 :(int) occurence of number given
    ------------
    Returns: dictionary: key = int value, value = count of how often key 
    appears in list  
    """
    
    dict = {int(k): int(v) for k, v in freq_dict.items()}
    dict = {k: v for k, v in freq_dict.items() if k in [num1,num2]}

    return dict 


In [ ]:
def add_freq_occurences(freq_dict):
    """Adds the amount of occurences for each key value within the 
    dictionary.
    -------------
    Parameters: 
        freq_dict (dcitionary): adjacency matrix for one group condition
    -------------
    Returns: total: (int) count of how many values in a key value pair are in the 
    dictionary 
     
    """
    total = 0 
    for key in freq_dict:
        total += freq_dict[key]

    return total 


In [ ]:
def compute_perc(count,total):
    """
    Makes a dictionary of the occurences of values within a list
    -----
    Parameters: 
    matrix1: (array) adjacency matrix for one group condition
    -----
    Returns: freq: (dictionary) key = int value, value = count of how often key 
    appears in list  
    """  

    freq = (count/total)*100   
    
    return freq 

In [ ]:
def comp_matrices(adj_matrix1, adj_matrix2):
    """
    Compares two adjacency matrices. Analyzes how proteins
    are correlated from one group to another.  
    --------------
    Parameters: 
    adj_matrix1: (array) adjacency matrix for one group condition
    adj_matrix2: (array) adjacency matrix for a different group condition
    --------------
    Prints: Outcome of comparing matricy (adding and subtracting) in frequency int
    """

    #### creating dictionaries and finding counts 
    # 1 = 1-->0, -1 = 0-->1
    # 2 = 1-->1, 0 = 0--> 0 

    sub = sub_matrices(adj_matrix1, adj_matrix2) 
    sub_flat = flatten_matrix(sub)
    sub_dict = retrieve_occurences(sub_flat)
    sub_dict = freq_dict(sub_dict,1,-1)

    add = add_matrices(adj_matrix1, adj_matrix2)
    add_flat = flatten_matrix(add)
    add_dict = retrieve_occurences(add_flat)
    add_dict = freq_dict(add_dict,0,2)

    comb_dict = combine_dict(sub_dict, add_dict)
    total = add_freq_occurences(comb_dict)

    occur_2 = comb_dict[2]
    occur_0 = comb_dict[0]
    occur_1 = comb_dict[1]
    occur_m1 = comb_dict[-1]

    freq_2  = compute_perc(occur_2, total)  
    freq_0  = compute_perc(occur_0, total)  
    freq_1  = compute_perc(occur_1, total)    
    freq_m1 = compute_perc(occur_m1, total) 

    print("Frequency of 1-->1:", f"{freq_2:.2f}")
    print("Frequency of 0-->0:", f"{freq_0:.2f}")
    print("Frequency of 0-->1:", f"{freq_1:.2f}")
    print("Frequency of 1-->0:", f"{freq_m1:.2f}")


* Runs comparison of adjacency matrices

In [ ]:
def pairwise_comp(control, experimental, file1, file2, corr_exp_vs_control, threshold=0.8): 
    """
    Compares TLCC output protein expressions for a control and experiemntal 
    group
    --------------
    Parameters: 
    control: (String) First group condition 
    experimental: (String) Second group condition to be compared to control group
    file1: (xcel) control raw xcel sheet data, must include fitted data 
    file2: (xcel) experimental raw xcel sheet data, must include fitted data 
    corr_exp_vs_control: (TLCC output csv) file consisting of protein comparison to itself across control versus experimental TLCC comparison
    threshold: (int) threshold to designate protein co-expression
    --------------
    Prints: Outcome of comparing matricy (adding and subtracting) in frequency int
    """
    
    file1 = pd.read_excel(file1)
    file2 = pd.read_excel(file2)

    df1 = file1[[col for col in file1 if col.startswith('Fitted')]] #Control
    df2 = file2[[col for col in file2 if col.startswith('Fitted')]] #Experimental 

    matrix1 = create_corr_matrix(df1,corr_exp_vs_control) # Control vs. Experimental w/Correlation data
    matrix2 = create_corr_matrix(df2,corr_exp_vs_control) # Control vs. Experimental w/Correlation data 

    adjacency_matrix1 = corr_to_adj(matrix1, threshold) #Control
    adjacency_matrix2 = corr_to_adj(matrix2, threshold) #Experimental

    print(control,"VS", experimental)
    comp_matrices(adjacency_matrix1, adjacency_matrix2) 


# Example Usage/Tutorial for ECHO datasets

In [ ]:
control = "Control Condition"
experimental = "Experimental Condition"
xlsx1 = "Control_Group.xlsx" #raw data
xlsx2 = "Experimental_Group.xlsx" #raw data
corr_exp_vs_control = "corr_exp_vs_control.csv" #Needs TLCC output to extract protein correlation against itself in control vs exp group comparison
threshold = 0.8 #0.8 = default, determines whether rythmicity same or different throughout condition

pairwise_comp(control, experimental, xlsx1, xlsx2, corr_exp_vs_control, threshold)